# Train Indoor Segmentation Model

This notebook trains a PointNet model for indoor scene segmentation.

**Classes:**
- 0: Floor
- 1: Ceiling
- 2: Wall
- 3: Object

## Steps:
1. Upload your training data (JSON files from the app)
2. Run all cells
3. Download the trained model

In [ ]:
# Install dependencies
!pip install -q torch numpy matplotlib tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import os
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Upload training data
from google.colab import files
print("Upload your training JSON files (exported from the app):")
uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} files")

In [ ]:
# Dataset class
class IndoorPointCloudDataset(Dataset):
    def __init__(self, json_files, num_points=4096, augment=True):
        self.samples = []
        self.num_points = num_points
        self.augment = augment
        
        for filepath in json_files:
            with open(filepath, 'r') as f:
                data = json.load(f)
            
            # Handle different export formats
            if 'points' in data and isinstance(data['points'], list):
                if isinstance(data['points'][0], dict):
                    # TrainingSample format
                    points = np.array([[p['x'], p['y'], p['z'], p['nx'], p['ny'], p['nz']] 
                                       for p in data['points']], dtype=np.float32)
                    labels = np.array([p['label'] for p in data['points']], dtype=np.int64)
                else:
                    # Numpy format
                    points = np.array(data['points'], dtype=np.float32)
                    labels = np.array(data['labels'], dtype=np.int64)
                
                if len(points) > 100:
                    self.samples.append((points, labels))
                    print(f"Loaded {filepath}: {len(points)} points")
        
        print(f"\nTotal samples: {len(self.samples)}")
    
    def __len__(self):
        return len(self.samples) * 10  # Augment by sampling multiple times
    
    def __getitem__(self, idx):
        sample_idx = idx % len(self.samples)
        points, labels = self.samples[sample_idx]
        
        # Random sampling
        if len(points) >= self.num_points:
            choice = np.random.choice(len(points), self.num_points, replace=False)
        else:
            choice = np.random.choice(len(points), self.num_points, replace=True)
        
        points = points[choice]
        labels = labels[choice]
        
        # Augmentation
        if self.augment:
            # Random rotation around Y axis
            theta = np.random.uniform(0, 2 * np.pi)
            cos_t, sin_t = np.cos(theta), np.sin(theta)
            rotation = np.array([[cos_t, 0, sin_t], [0, 1, 0], [-sin_t, 0, cos_t]])
            points[:, :3] = points[:, :3] @ rotation.T
            points[:, 3:6] = points[:, 3:6] @ rotation.T
            
            # Random jitter
            points[:, :3] += np.random.normal(0, 0.01, (self.num_points, 3))
            
            # Random scale
            scale = np.random.uniform(0.9, 1.1)
            points[:, :3] *= scale
        
        return torch.from_numpy(points), torch.from_numpy(labels)

In [ ]:
# PointNet Model
class PointNetSegmentation(nn.Module):
    def __init__(self, num_classes=4, input_channels=6):
        super().__init__()
        self.num_classes = num_classes
        
        # Encoder
        self.enc1 = nn.Conv1d(input_channels, 64, 1)
        self.enc2 = nn.Conv1d(64, 128, 1)
        self.enc3 = nn.Conv1d(128, 256, 1)
        self.enc4 = nn.Conv1d(256, 512, 1)
        self.enc5 = nn.Conv1d(512, 1024, 1)
        
        # Batch normalization
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(256)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(1024)
        
        # Decoder
        self.dec1 = nn.Conv1d(1024 + 64, 512, 1)
        self.dec2 = nn.Conv1d(512, 256, 1)
        self.dec3 = nn.Conv1d(256, 128, 1)
        self.dec4 = nn.Conv1d(128, num_classes, 1)
        
        self.dbn1 = nn.BatchNorm1d(512)
        self.dbn2 = nn.BatchNorm1d(256)
        self.dbn3 = nn.BatchNorm1d(128)
        
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        batch_size = x.size(0)
        num_points = x.size(1)
        x = x.transpose(2, 1)  # (B, 6, N)
        
        # Encode
        x1 = F.relu(self.bn1(self.enc1(x)))
        x2 = F.relu(self.bn2(self.enc2(x1)))
        x3 = F.relu(self.bn3(self.enc3(x2)))
        x4 = F.relu(self.bn4(self.enc4(x3)))
        x5 = F.relu(self.bn5(self.enc5(x4)))
        
        # Global feature
        global_feat = torch.max(x5, 2, keepdim=True)[0]
        global_feat = global_feat.repeat(1, 1, num_points)
        
        # Decode with skip connection
        x = torch.cat([global_feat, x1], dim=1)
        x = F.relu(self.dbn1(self.dec1(x)))
        x = self.dropout(x)
        x = F.relu(self.dbn2(self.dec2(x)))
        x = self.dropout(x)
        x = F.relu(self.dbn3(self.dec3(x)))
        x = self.dec4(x)
        
        x = x.transpose(2, 1)  # (B, N, C)
        return x  # Logits, not softmax

print("Model architecture defined")

In [ ]:
# Load data
json_files = list(uploaded.keys())
dataset = IndoorPointCloudDataset(json_files, num_points=4096)

# Split into train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# Initialize model
model = PointNetSegmentation(num_classes=4, input_channels=6).to(device)

# Optionally load pretrained weights
# if os.path.exists('IndoorSegmentation.pt'):
#     checkpoint = torch.load('IndoorSegmentation.pt', map_location=device)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     print("Loaded pretrained weights")

# Loss with class weights (handle imbalanced data)
class_weights = torch.tensor([1.0, 1.0, 1.5, 2.0]).to(device)  # Weight objects more
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training loop
NUM_EPOCHS = 50
best_val_acc = 0
train_losses = []
val_accs = []

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    total_loss = 0
    
    for points, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        points = points.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(points)  # (B, N, 4)
        
        loss = criterion(outputs.reshape(-1, 4), labels.reshape(-1))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Validate
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for points, labels in val_loader:
            points = points.to(device)
            labels = labels.to(device)
            
            outputs = model(points)
            _, predicted = outputs.max(dim=2)
            
            total += labels.numel()
            correct += (predicted == labels).sum().item()
    
    val_acc = correct / total
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Val Acc={val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'num_classes': 4,
            'input_channels': 6,
            'epoch': epoch,
            'val_acc': val_acc
        }, 'IndoorSegmentation_best.pt')
        print(f"  -> Saved best model (acc={val_acc:.4f})")
    
    scheduler.step()

print(f"\nTraining complete! Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(val_accs)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()

In [ ]:
# Per-class accuracy
model.eval()
class_correct = [0] * 4
class_total = [0] * 4
class_names = ['Floor', 'Ceiling', 'Wall', 'Object']

with torch.no_grad():
    for points, labels in val_loader:
        points = points.to(device)
        labels = labels.to(device)
        
        outputs = model(points)
        _, predicted = outputs.max(dim=2)
        
        for c in range(4):
            mask = labels == c
            class_total[c] += mask.sum().item()
            class_correct[c] += ((predicted == labels) & mask).sum().item()

print("\nPer-class accuracy:")
for i, name in enumerate(class_names):
    if class_total[i] > 0:
        acc = class_correct[i] / class_total[i]
        print(f"  {name}: {acc:.4f} ({class_correct[i]}/{class_total[i]})")
    else:
        print(f"  {name}: No samples")

In [ ]:
# Save final model for Core ML conversion
model.eval()

# Load best weights
checkpoint = torch.load('IndoorSegmentation_best.pt')
model.load_state_dict(checkpoint['model_state_dict'])

# Create export version (without batch norm in eval mode)
class PointNetExport(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
    
    def forward(self, x):
        logits = self.model(x)
        return F.softmax(logits, dim=-1)

export_model = PointNetExport(model)
export_model.eval()

# Save for Core ML conversion
torch.save({
    'model_state_dict': model.state_dict(),
    'num_classes': 4,
    'input_channels': 6,
    'best_val_acc': best_val_acc
}, 'IndoorSegmentation.pt')

print("Model saved as IndoorSegmentation.pt")

In [ ]:
# Convert to Core ML (if running on macOS or Colab)
!pip install -q coremltools

import coremltools as ct

# Trace model
example_input = torch.randn(1, 4096, 6)
traced = torch.jit.trace(export_model.cpu(), example_input)

# Convert
mlmodel = ct.convert(
    traced,
    inputs=[
        ct.TensorType(
            name="points",
            shape=(1, ct.RangeDim(100, 100000, 4096), 6),
            dtype=np.float32
        )
    ],
    outputs=[
        ct.TensorType(name="classifications", dtype=np.float32)
    ],
    minimum_deployment_target=ct.target.iOS15,
    compute_precision=ct.precision.FLOAT16,
)

# Metadata
mlmodel.author = "LiDAR Scanner - Custom Trained"
mlmodel.short_description = "Indoor scene semantic segmentation (custom trained)"
mlmodel.version = "2.0"

mlmodel.save("IndoorSegmentation.mlpackage")
print("Core ML model saved!")

In [ ]:
# Download trained models
!zip -r IndoorSegmentation.mlpackage.zip IndoorSegmentation.mlpackage

files.download('IndoorSegmentation.pt')
files.download('IndoorSegmentation.mlpackage.zip')
files.download('training_curves.png')

print("\nDownload complete!")
print("Copy IndoorSegmentation.mlpackage to your Xcode project's LiDARScanner/ML/ folder")